In [14]:
from datasets import Dataset 
from dotenv import load_dotenv
import os
from pathlib import Path
from ragas import evaluate
from ragas.metrics import (faithfulness , answer_relevancy , context_recall,context_precision )
from ragas.llms import LangchainLLMWrapper
from langchain_ollama import ChatOllama
import json
import sys
sys.path.insert(0,'..')
from src.graph.graph import graph


/tmp/ipykernel_603792/181577914.py:6: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (faithfulness , answer_relevancy , context_recall,context_precision )
/tmp/ipykernel_603792/181577914.py:6: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (faithfulness , answer_relevancy , context_recall,context_precision )
/tmp/ipykernel_603792/181577914.py:6: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import (fait

In [8]:
load_dotenv()
EVAL_DATASET =  Path(os.getenv('RISK_DATA_PATH'))



In [21]:
with open(Path(EVAL_DATASET)/'test_set.json','r') as f:
    test_data = json.load(f)

test_set = dict(list(test_data.items())[:25])


In [22]:
def raga_dataset(graph , test_data:dict):
    samples =[]
    errors =0 
    for i ,(key,value)  in enumerate(test_data.items()):
        result = graph.invoke({
            "input_document":value[1],
            "document_type":"financial",
            "compliance_questions": [],
            "retrieved_contexts": [],
            "reasoning_trace": "",
            "findings": [],
            "max_severity": "",
            "report_markdown": "",
            "audit_id": "",
            "escalated": False
        })
        contexts = []
        for ctx in result['retrieved_contexts']:
            for c in ctx.associated_chks:
                contexts.append(c.chunk_text)
        answer = "\n".join([
            f"{f.violation} ({f.severity}) - {f.reasoning}"
            for f in result["findings"]
        ])
        samples.append({
            "questions": value[1], "contexts":contexts , "answer":answer , "ground_truth":value[2]
        })
    return Dataset.from_list(samples)

In [23]:
compliled_graph = graph()

dataset = raga_dataset(compliled_graph,test_set)

STATE KEYS: dict_keys(['input_document', 'document_type', 'compliance_questions', 'retrieved_contexts', 'reasoning_trace', 'findings', 'max_severity', 'report_markdown', 'audit_id', 'escalated'])


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1049.04it/s]
BertModel LOAD REPORT from: aminhaeri/risk-embed
Key                 | Status  | 
--------------------+---------+-
pooler.dense.bias   | MISSING | 
pooler.dense.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


STATE KEYS: dict_keys(['input_document', 'document_type', 'compliance_questions', 'retrieved_contexts', 'reasoning_trace', 'findings', 'max_severity', 'report_markdown', 'audit_id', 'escalated'])


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 997.98it/s]
BertModel LOAD REPORT from: aminhaeri/risk-embed
Key                 | Status  | 
--------------------+---------+-
pooler.dense.bias   | MISSING | 
pooler.dense.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


OutOfMemoryError: CUDA out of memory. Tried to allocate 90.00 MiB. GPU 0 has a total capacity of 5.64 GiB of which 24.94 MiB is free. Including non-PyTorch memory, this process has 1.00 GiB memory in use. Process 1840207 has 4.54 GiB memory in use. Of the allocated memory 847.21 MiB is allocated by PyTorch, and 74.79 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
llm = ChatOllama(model ='llama3',temperature = 0)
ragas_llm = LangchainLLMWrapper(llm)

results = evaluate(
    dataset=dataset,metrics=[faithfulness,answer_relevancy,context_recall,context_precision] , llm=ragas_llm
)

In [ ]:
results

In [ ]:
import pandas as pd

df = results.to_pandas()

print(f"Faithfulness:      {df['faithfulness'].mean():.4f}")
print(f"Answer Relevancy:  {df['answer_relevancy'].mean():.4f}")
print(f"Context Recall:    {df['context_recall'].mean():.4f}")
print(f"Context Precision: {df['context_precision'].mean():.4f}")


In [ ]:
df.to_csv('../logs/ragas_results.csv', index=False)